In [28]:
import numpy as np
import pandas as pd
from math import sqrt
from functools import partial

In [29]:
texts_df = pd.read_pickle('../data/dfs/texts.pkl')
texts_df.head()

,text,bbox,page,doc_id
0,УДК [37.016:94(477)](075.3) Г51,"(73.70100402832031, 61.3690185546875, 214.3210...",1,0
1,Рекомендовано Міністерством освіти і науки Укр...,"(100.1719970703125, 121.24398803710938, 398.73...",1,0
2,Видано за рахунок державних коштів. Продаж заб...,"(136.25099182128906, 150.72097778320312, 362.6...",1,0
3,Гісем О. В. Г51 Історія України (рівень станда...,"(73.69699096679688, 352.01898193359375, 427.52...",1,0
4,УДК [37.016:94(477)](075.3),"(314.5790100097656, 417.760986328125, 425.2031...",1,0


In [30]:
texts_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30578 entries, 0 to 30577
Data columns (total 4 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   text    30578 non-null  object
 1   bbox    30578 non-null  object
 2   page    30578 non-null  int64 
 3   doc_id  30578 non-null  int64 
dtypes: int64(2), object(2)
memory usage: 955.7+ KB


In [31]:
images_df = pd.read_pickle('../data/dfs/images.pkl')
images_df.head()

,image_path,bbox,page,doc_id,filtered
0,doc0_page0_0.jpeg,"(0.0, 0.0, 467.760009765625, 595.2000122070312)",0,0,True
1,doc0_page1_6.png,"(84.24800109863281, 447.7869567871094, 132.744...",1,0,False
2,doc0_page9_8.png,"(37.176998138427734, 62.65498352050781, 220.77...",9,0,True
3,doc0_page9_9.png,"(240.15599060058594, 61.83399200439453, 425.95...",9,0,True
4,doc0_page11_5.png,"(22.917999267578125, 437.96197509765625, 127.3...",11,0,True


In [38]:
images_df = images_df[images_df['filtered']]

In [39]:
images_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2850 entries, 0 to 5167
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   image_path  2850 non-null   object
 1   bbox        2850 non-null   object
 2   page        2850 non-null   int64 
 3   doc_id      2850 non-null   int64 
 4   filtered    2850 non-null   bool  
 5   caption     2823 non-null   object
dtypes: bool(1), int64(2), object(3)
memory usage: 136.4+ KB


In [33]:
def find_centroid(bbox: tuple):
    return bbox[2] - bbox[0] / 2, bbox[3] - bbox[1] / 2

def get_distance(first_bbox: tuple, second_bbox: tuple):
    first_centroid = find_centroid(first_bbox)
    second_centroid = find_centroid(second_bbox)

    return sqrt((second_centroid[0] - first_centroid[0]) ** 2
                + (second_centroid[1] - first_centroid[0]) ** 2)

In [34]:
def link_image_to_caption():
    for index, image_row in images_df.iterrows():
        texts_candidates_df = texts_df[
            (texts_df['doc_id'] == image_row['doc_id']) &
            (texts_df['page'] == image_row['page'])
        ]


        texts_candidates_df = texts_candidates_df[
            texts_candidates_df['text'].str.len() >= 5
        ]

        if texts_candidates_df.empty:
            continue

        distance_criteria = lambda second_bbox: get_distance(
            first_bbox=image_row['bbox'],
            second_bbox=second_bbox
        )

        idx = texts_candidates_df['bbox'].apply(distance_criteria).idxmin()
        images_df.loc[index, 'caption'] = texts_candidates_df.loc[idx, 'text']


In [40]:
images_df['caption'] = None
link_image_to_caption()

In [41]:
images_df.iloc[0:2850]

,image_path,bbox,page,doc_id,filtered,caption
0,doc0_page0_0.jpeg,"(0.0, 0.0, 467.760009765625, 595.2000122070312)",0,0,True,None
2,doc0_page9_8.png,"(37.176998138427734, 62.65498352050781, 220.77...",9,0,True,"Репресії — заходи державного примусу, по- кара..."
3,doc0_page9_9.png,"(240.15599060058594, 61.83399200439453, 425.95...",9,0,True,"3 Ставлення до ­війни політичних сил, що діяли..."
4,doc0_page11_5.png,"(22.917999267578125, 437.96197509765625, 127.3...",11,0,True,Розділ I
5,doc0_page13_8.png,"(22.909000396728516, 419.3099365234375, 158.50...",13,0,True,Російські солдати радіють пере- мозі в Галицьк...
...,...,...,...,...,...,...
5163,doc10_page316_18.jpeg,"(354.33099365234375, 452.0369873046875, 533.97...",316,10,True,Зруйнований 4-й енергоблок Чорнобильської АЕС ...
5164,doc10_page317_17.jpeg,"(34.26580047607422, 75.10700988769531, 212.602...",317,10,True,Виступ активіста на загальних зборах працівник...
5165,doc10_page317_19.jpeg,"(34.06494903564453, 447.7210998535156, 212.466...",317,10,True,Виступ активіста на загальних зборах працівник...
5166,doc10_page318_14.jpeg,"(354.5364990234375, 65.01054382324219, 532.819...",318,10,True,Фільтраційний пункт — тут: місце перевірки рос...
